In [1]:
%run code/losses.py

In [2]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

# from losses import compute_fkl, compute_rkl
from datasets import load_from_disk


max_seq_length = 2048
dtype = None 
load_in_4bit = True

max_new_tokens = 128
temperature = 2.0

class OPDTrainer(SFTTrainer):
    def __init__(self, *args, teacher_model=None, max_new_tokens=128, temp=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.max_new_tokens = max_new_tokens
        self.temp = temp
        self.tokenizer = kwargs.get("processing_class", kwargs.get("tokenizer"))

    def _generate_on_policy(self, model, input_ids, attention_mask):
        was_training = model.training
        model.eval()
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.8,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                use_cache=True,
            )
        if was_training:
            model.train()
        return generated_ids

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        prompt_input_ids = inputs["input_ids"]
        if "attention_mask" in inputs:
            prompt_attention_mask = inputs["attention_mask"]
        else:
            prompt_attention_mask = prompt_input_ids.ne(self.tokenizer.pad_token_id).long()
            
        prompt_lengths = prompt_attention_mask.sum(dim=1)

        # 1. 学生模型自行生成轨迹
        generated_ids = self._generate_on_policy(
            model, prompt_input_ids, prompt_attention_mask
        )
        generated_attention_mask = generated_ids.ne(self.tokenizer.pad_token_id).long()

        # 2. 构建 labels：把 prompt 部分的 token 设为 -100
        labels = generated_ids.clone()
        for row_idx, prompt_len in enumerate(prompt_lengths):
            labels[row_idx, :prompt_len] = -100
        labels = labels.masked_fill(generated_attention_mask.eq(0), -100)

        # 3. GKD ：只拿 logits，不计算自带 SFT Loss
        outputs_student = model(
            input_ids=generated_ids,
            attention_mask=generated_attention_mask,
            return_dict=True
        )
        logits = outputs_student.logits

        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=generated_ids,
                attention_mask=generated_attention_mask,
                return_dict=True
            )
            teacher_logits = teacher_outputs.logits

        kl = 0
        if isinstance(logits, torch.Tensor) and isinstance(teacher_logits, torch.Tensor):
            if logits.shape[-1] != teacher_logits.shape[-1]:
                teacher_logits = teacher_logits[:, :, :logits.shape[-1]]
            kl = compute_rkl(logits, teacher_logits, labels, padding_id=-100, temp=self.temp)
            valid_tokens = labels.ne(-100).sum().clamp_min(1)
            kl = kl / valid_tokens

        loss_total = kl
        return (loss_total, outputs_student) if return_outputs else loss_total

# ------------------------------------------------------------------
# 1. 初始化模型
print("1. 初始化全新的 Student 和 Teacher 模型...")
student, _ = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

student = FastLanguageModel.get_peft_model(
    student,
    r=16, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0, 
    bias="none",    
    use_gradient_checkpointing="unsloth", 
    random_state=3407,
)

teacher, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen_teacher_finetune",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(teacher)
teacher.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# ------------------------------------------------------------------
# 2. 课程学习 (Curriculum Learning) 训练流
stages = ["easy", "medium", "hard", "full"]

for stage_idx, stage_name in enumerate(stages):
    print(f"\n{'='*50}")
    print(f" 开始课程学习阶段 {stage_idx + 1}/4 : [{stage_name.upper()}] 数据集")
    print(f"{'='*50}")

    # 加载当前阶段的数据集
    current_dataset = load_from_disk(f"./data_splits/data_{stage_name}")
    print(f"当前阶段数据量: {len(current_dataset)} 条")

    # 每个阶段跑 1 个 Epoch，输出到独立文件夹防止日志冲突
    args = TrainingArguments(
        output_dir=f'./results_cl_opd_{stage_name}',
        num_train_epochs=1, 
        do_train=True,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=16,
        logging_steps=10,
        save_strategy='no', # 中途不保存，最后统一保存
        bf16=True,
        learning_rate=0.0005,
        lr_scheduler_type='constant', # 保持常数学习率，跨阶段切换时最平滑
        optim="adamw_torch_fused",
        remove_unused_columns=False,
    )

    # 每次新建一个 Trainer 实例以刷新 DataLoader，但底层 model 权重在内存中是连续继承的
    trainer = OPDTrainer(
        model=student,
        teacher_model=teacher,
        processing_class=tokenizer,
        train_dataset=current_dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        args=args,
        max_new_tokens=max_new_tokens,
        temp=temperature,
    )

    # 动态移除 Notebook 进度条以防崩溃
    '''callbacks_to_remove = []
    for callback in trainer.callback_handler.callbacks:
        if "NotebookProgressCallback" in str(type(callback)):
            callbacks_to_remove.append(callback)
    for callback in callbacks_to_remove:
        trainer.remove_callback(callback)'''

    import warnings
    import transformers
    # 强行关闭 HuggingFace 生成时的各种警告
    transformers.logging.set_verbosity_error()
    warnings.filterwarnings("ignore")

    # 启动本阶段训练
    trainer.train(resume_from_checkpoint=False)


# ------------------------------------------------------------------
# 3. 保存最终的 CL 模型
print("\n 所有的课程学习阶段已全部完成，正在保存模型...")
student.save_pretrained("qwen_student_cl_opd")
tokenizer.save_pretrained("qwen_student_cl_opd")
print(" 模型已保存至 qwen_student_cl_opd 文件夹！")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
1. 初始化全新的 Student 和 Teacher 模型...
==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.8.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

{'loss': '4.25', 'grad_norm': '1.79', 'learning_rate': '0.0005', 'epoch': '0.3131'}
{'loss': '3.825', 'grad_norm': '1.359', 'learning_rate': '0.0005', 'epoch': '0.6262'}
{'loss': '3.623', 'grad_norm': '1.056', 'learning_rate': '0.0005', 'epoch': '0.9393'}
{'train_runtime': '2804', 'train_samples_per_second': '0.365', 'train_steps_per_second': '0.011', 'train_loss': '3.87', 'epoch': '1'}

 开始课程学习阶段 3/4 : [HARD] 数据集
当前阶段数据量: 1500 条


Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/1500 [00:00<?, ? examples/s]

{'loss': '3.648', 'grad_norm': '2.125', 'learning_rate': '0.0005', 'epoch': '0.2133'}
{'loss': '3.554', 'grad_norm': '1.137', 'learning_rate': '0.0005', 'epoch': '0.4267'}
{'loss': '3.412', 'grad_norm': '1.199', 'learning_rate': '0.0005', 'epoch': '0.64'}
{'loss': '3.401', 'grad_norm': '0.8413', 'learning_rate': '0.0005', 'epoch': '0.8533'}
{'train_runtime': '3975', 'train_samples_per_second': '0.377', 'train_steps_per_second': '0.012', 'train_loss': '3.47', 'epoch': '1'}

 开始课程学习阶段 4/4 : [FULL] 数据集
当前阶段数据量: 2000 条


Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/2000 [00:00<?, ? examples/s]

{'loss': '3.4', 'grad_norm': '0.6684', 'learning_rate': '0.0005', 'epoch': '0.16'}
{'loss': '3.268', 'grad_norm': '0.5779', 'learning_rate': '0.0005', 'epoch': '0.32'}
{'loss': '3.312', 'grad_norm': '1.742', 'learning_rate': '0.0005', 'epoch': '0.48'}
{'loss': '3.264', 'grad_norm': '1.193', 'learning_rate': '0.0005', 'epoch': '0.64'}
{'loss': '3.237', 'grad_norm': '0.7911', 'learning_rate': '0.0005', 'epoch': '0.8'}
{'loss': '3.191', 'grad_norm': '0.8996', 'learning_rate': '0.0005', 'epoch': '0.96'}
{'train_runtime': '5317', 'train_samples_per_second': '0.376', 'train_steps_per_second': '0.012', 'train_loss': '3.252', 'epoch': '1'}

 所有的课程学习阶段已全部完成，正在保存模型...
 模型已保存至 qwen_student_cl_opd 文件夹！


下面计算kl散度

In [1]:
%run code/losses.py

In [2]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer


# from losses import compute_fkl, compute_rkl
from datasets import load_from_disk



max_seq_length = 2048
load_in_4bit = True
max_new_tokens = 128
temperature = 2.0


class OPDTrainer(SFTTrainer):
    def __init__(self, *args, teacher_model=None, max_new_tokens=128, temp=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.max_new_tokens = max_new_tokens
        self.temp = temp
        self.tokenizer = kwargs.get("processing_class", kwargs.get("tokenizer"))

    def _generate_on_policy(self, model, input_ids, attention_mask):
        was_training = model.training
        model.eval()
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.8,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                use_cache=True,
            )
        if was_training:
            model.train()
        return generated_ids

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        prompt_input_ids = inputs["input_ids"]
        prompt_attention_mask = inputs.get("attention_mask", prompt_input_ids.ne(self.tokenizer.pad_token_id).long())
        prompt_lengths = prompt_attention_mask.sum(dim=1)

        generated_ids = self._generate_on_policy(model, prompt_input_ids, prompt_attention_mask)
        generated_attention_mask = generated_ids.ne(self.tokenizer.pad_token_id).long()

        labels = generated_ids.clone()
        for row_idx, prompt_len in enumerate(prompt_lengths):
            labels[row_idx, :prompt_len] = -100
        labels = labels.masked_fill(generated_attention_mask.eq(0), -100)

        outputs_student = model(
            input_ids=generated_ids,
            attention_mask=generated_attention_mask,
            return_dict=True
        )
        logits = outputs_student.logits

        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=generated_ids,
                attention_mask=generated_attention_mask,
                return_dict=True
            )
            teacher_logits = teacher_outputs.logits

        kl = 0
        if isinstance(logits, torch.Tensor) and isinstance(teacher_logits, torch.Tensor):
            if logits.shape[-1] != teacher_logits.shape[-1]:
                teacher_logits = teacher_logits[:, :, :logits.shape[-1]]
            kl = compute_rkl(logits, teacher_logits, labels, padding_id=-100, temp=self.temp)
            valid_tokens = labels.ne(-100).sum().clamp_min(1)
            kl = kl / valid_tokens

        loss_total = kl
        return (loss_total, outputs_student) if return_outputs else loss_total


# ---------------------------------------------------------
# 加载模型与数据

print("1. 正在加载【课程学习】训练完成的 Student 模型 (qwen_student_cl_opd)...")
student, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen_student_cl_opd", #  CL 训练出来的模型
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(student)
student.eval()

print("2. 正在加载 Teacher 模型 (qwen_teacher_finetune)...")
teacher, _ = FastLanguageModel.from_pretrained(
    model_name="qwen_teacher_finetune",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(teacher)
teacher.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("3. 加载测试集...")
test_dataset = load_from_disk("./data_splits/data_test")
print(f" 测试集加载成功，共 {len(test_dataset)} 条数据")

# ---------------------------------------------------------
#  启动评估

args = TrainingArguments(
    output_dir='./eval_results_cl',
    per_device_eval_batch_size=2,
    report_to="none"
)

trainer = OPDTrainer(
    model=student,
    teacher_model=teacher,
    processing_class=tokenizer,
    train_dataset=test_dataset,   # 骗过框架检查
    eval_dataset=test_dataset,    # 真正评估用的是这个
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=args,
    max_new_tokens=max_new_tokens,
    temp=temperature,
)

import warnings
import transformers
# 强行关闭 HuggingFace 各种警告
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

print("\n 开始在测试集上计算 KL 散度...")

# 动态移除导致崩溃的 HTML 进度条，换成纯文本 tqdm
from transformers.trainer_callback import ProgressCallback
callbacks_to_remove = []
for callback in trainer.callback_handler.callbacks:
    if "NotebookProgressCallback" in str(type(callback)):
        callbacks_to_remove.append(callback)
for callback in callbacks_to_remove:
    trainer.remove_callback(callback)
trainer.add_callback(ProgressCallback)

metrics = trainer.evaluate()

print("\n" + "="*50)
print(f" 最终评估结果 (Test Set KL Divergence): {metrics['eval_loss']:.4f}")
print("="*50)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
1. 正在加载【课程学习】训练完成的 Student 模型 (qwen_student_cl_opd)...
==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.8.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


2. 正在加载 Teacher 模型 (qwen_teacher_finetune)...
==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

3. 加载测试集...
 测试集加载成功，共 200 条数据


Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/200 [00:00<?, ? examples/s]


 开始在测试集上计算 KL 散度...


  0%|          | 0/100 [00:00<?, ?it/s]


 最终评估结果 (Test Set KL Divergence): 0.2077
